# Tutorial 02e · El mismo analisis, tres veces

**Este es el "dolor" del tutorial.** Vamos a calcular una metrica trivial —el
precio promedio ponderado de una tienda ficticia— de **tres** maneras y vamos
a comprobar que el numero debe ser el mismo.

Lo que cambia no es el resultado: es quien puede reproducirlo.

| Estado | Como se ejecuta | Que falla |
|---|---|---|
| **Caotico** | Celdas fuera de orden, `!pip install` a mitad, constantes reescritas a mano | el resultado depende del orden de Shift+Enter |
| **Script** | `python analisis.py` de arriba abajo | reproducible, pero no se puede importar ni testear por partes |
| **Paquete** | `from analisis_ventas import ...` | importable, testeable, versionado |

El numero tiene que salir igual en los tres.

In [1]:
# Punto de partida comun a los tres estados. Generamos el dataset una vez,
# lo guardamos en un parquet temporal y los tres estados lo van a leer de alli.
# NO es parte de la demostracion, solo asegura que los tres leen el mismo dato.
from pathlib import Path
from datos_sinteticos import generar_datos

RAIZ = Path.cwd()
DATOS = RAIZ / "ventas.parquet"

datos = generar_datos()
datos.to_parquet(DATOS, index=False)
print(f"shape: {datos.shape}")
print(f"parquet escrito en: {DATOS.name}")
datos.head()

shape: (5000, 4)
parquet escrito en: ventas.parquet


,Invoice,Product,Price,Quantity
0,A89267,D,3.82,6
1,A58646,B,5.34,4
2,A72529,D,500.00,6
3,A70930,C,5.07,9
4,A24927,A,6.25,5


---

## Estado 1 - El notebook caotico

Lo que viene es una reconstruccion fiel de un notebook real. Tiene **cuatro
problemas**, y ninguno de los cuatro produce un error. Antes de ejecutar nada,
intenta encontrarlos:

1. una dependencia instalada a mitad del notebook, que no queda declarada en
   ningun archivo;
2. constantes de negocio (rango valido de precios) escritas a mano, con
   **dos valores distintos** entre dos celdas;
3. una celda que **relee los datos** y otra que no, de modo que el orden en
   que se ejecutan cambia el resultado;
4. ninguna indicacion de que las facturas con prefijo "C" son cancelaciones
   (algunas celdas las filtran, otras no).

In [2]:
# --- CELDA 12 del notebook original: pip install a mitad del analisis ---
# ANTI-PATRON. Se deja aqui a proposito, COMENTADO, porque ejecutarlo
# modificaria tu entorno.
#
#   !pip install pandas==1.5.3
#
# Tres preguntas que nadie puede responder mirando este notebook:
#   - en QUE entorno instalo (el del kernel? otro?)
#   - por que 1.5.3, y quien lo decidio
#   - donde queda registrado para que otra persona reproduzca
print("linea 1 del anti-patron: no la ejecutes, solo mirala.")

linea 1 del anti-patron: no la ejecutes, solo mirala.


In [3]:
# --- Celda A: cargar ---
import pandas as pd
df = pd.read_parquet(DATOS)
print(f"{len(df)} filas")

5000 filas


In [4]:
# --- Celda B: "limpieza". Constantes escritas a mano, version 1 ---
# Rango de precios valido escrito a mano: 0.01 - 100
# Descarta cancelaciones (facturas con "C")
df_b = df[(df.Price >= 0.01) & (df.Price <= 100) & (~df.Invoice.str.startswith("C"))]
precio_promedio_b = (df_b.Price * df_b.Quantity).sum() / df_b.Quantity.sum()
print(f"precio promedio ponderado (B): {precio_promedio_b:.4f}")

precio promedio ponderado (B): 8.9240


In [5]:
# --- Celda C: la "misma" limpieza, con OTROS limites. Version 2 ---
# Aqui esta el bug real: alguien probo otros umbrales de precio, no los
# revirtio, Y ademas olvido filtrar cancelaciones. Y esta celda RELEE el
# parquet (reinicia df) mientras que la celda B lo acumulaba. Esa asimetria
# —una celda que reinicia junto a otra que acumula— es lo que hace que el
# orden de Shift+Enter importe.
df = pd.read_parquet(DATOS)  # <- relee, resetea estado
df_c = df[(df.Price >= 0.1) & (df.Price <= 1000)]  # <- otros limites, no filtra C
precio_promedio_c = (df_c.Price * df_c.Quantity).sum() / df_c.Quantity.sum()
print(f"precio promedio ponderado (C): {precio_promedio_c:.4f}")
print(f"\ndiferencia B vs C: {abs(precio_promedio_b - precio_promedio_c):.4f}")

precio promedio ponderado (C): 13.1596

diferencia B vs C: 4.2356


### Lo que acaba de pasar

Las celdas B y C hacen "lo mismo" y dan **numeros distintos**, porque los
limites de precio valido —una **decision de negocio**— estan escritos a mano
en dos sitios. No hay un error en pantalla. Hay dos verdades.

Pero el problema de fondo es peor y esta en la asimetria entre las dos celdas:

| | Relee el parquet | Rango | Filtra cancelaciones | Efecto |
|---|---|---|---|---|
| **B** | no | [0.01, 100] | si | acumula sobre el estado anterior |
| **C** | si | [0.1, 1000] | no | reinicia y aplica otras reglas |

**Si corres B y luego C**: dos numeros distintos.
**Si corres C y luego B**: B hereda el df reiniciado + sin cancelaciones
filtradas → sale otro numero MAS. Son al menos tres resultados posibles.

Un notebook caotico es, literalmente, un sistema **no determinista**.

---

## Estado 2 - El script

Mismo calculo, en un archivo de texto que se ejecuta de arriba abajo. Es una
mejora **real**, no cosmetica:

- el orden de ejecucion es el orden del archivo. Ya no hay estado oculto.
- las constantes estan arriba, en un solo sitio.
- se puede correr en CI: `python analisis.py`.

Lo que todavia **no** se puede hacer:

- importar una parte para testearla por separado (`from analisis import limpiar`).
- reutilizar las constantes desde otro script sin copiarlas.
- declarar dependencias con un `pyproject.toml`.

Es un paso intermedio honesto, no una solucion.

In [6]:
# Escribimos el script a un archivo temporal y lo ejecutamos como lo haria
# un CI o un cron. No se commitea: es un paso intermedio de la demostracion.
import subprocess
import sys
import tempfile
import textwrap

SCRIPT = textwrap.dedent(f'''
    # analisis.py - estado 2: todo en un archivo, orden explicito
    import pandas as pd
    from pathlib import Path

    # Constantes, en UN solo sitio arriba
    PRECIO_MIN = 0.01
    PRECIO_MAX = 100.0
    CANCELACION_PREFIX = "C"
    DATOS = Path("{DATOS}")

    def main():
        df = pd.read_parquet(DATOS)
        df = df[~df.Invoice.astype(str).str.startswith(CANCELACION_PREFIX)]
        df = df[df.Price.between(PRECIO_MIN, PRECIO_MAX)]
        precio = (df.Price * df.Quantity).sum() / df.Quantity.sum()
        print(f"precio promedio ponderado (script): {{precio:.4f}}")
        return precio

    if __name__ == "__main__":
        main()
''').lstrip()

with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
    f.write(SCRIPT)
    ruta_script = f.name

resultado = subprocess.run(
    [sys.executable, ruta_script], capture_output=True, text=True
)
print(resultado.stdout)
if resultado.returncode != 0:
    print("STDERR:", resultado.stderr)

precio promedio ponderado (script): 8.9240



---

## Estado 3 - El paquete instalable

El mismo calculo usando el paquete `analisis_ventas` que vive en
[`paquete/`](paquete/). Fijate en lo que **no** hay en las celdas siguientes:
ni la ruta del parquet, ni los limites de precio, ni el prefijo de
cancelacion. Todo eso vive en un solo sitio, [`paquete/src/analisis_ventas/config.py`](paquete/src/analisis_ventas/config.py),
y el resto lo importa.

Antes de poder importar, hay que **instalar** el paquete. Lo hacemos en modo
editable (`-e`) para que los cambios al codigo se reflejen al instante:

In [7]:
# Instalar el paquete del tutorial en modo editable.
# Esto corre UNA sola vez; si ya lo instalaste, pip dira "already satisfied".
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "paquete/", "--quiet"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    print("STDERR:", r.stderr)
else:
    print("paquete analisis_ventas instalado.")

STDERR: /Users/david.palacio/Documents/data-projects-lab/.venv/bin/python: No module named pip



In [8]:
# La forma final: importar las piezas que necesitas, componerlas.
import pandas as pd

from analisis_ventas import limpiar, precio_promedio_ponderado
from analisis_ventas.config import PRECIO_MIN, PRECIO_MAX, CANCELACION_PREFIX

print(f"decisiones de negocio (desde config.py):")
print(f"  rango de precio valido: [{PRECIO_MIN}, {PRECIO_MAX}]")
print(f"  prefijo de cancelacion: {CANCELACION_PREFIX!r}\n")

df = pd.read_parquet(DATOS)
limpio = limpiar(df)
precio = precio_promedio_ponderado(limpio)
print(f"precio promedio ponderado (paquete): {precio:.4f}")

decisiones de negocio (desde config.py):
  rango de precio valido: [0.01, 100.0]
  prefijo de cancelacion: 'C'

precio promedio ponderado (paquete): 8.9240


### El mismo numero, y tres propiedades nuevas

El resultado coincide con el del estado 2 (y con la celda B del estado 1,
la "version correcta"). Lo que ha cambiado es todo lo demas:

1. **Los limites son una decision, no una constante suelta.** Estan en
   `config.py` con un comentario que dice **por que**: *"fuera de ese rango
   son errores de captura, no ventas"*. Cambiarlos es un commit revisable.
2. **Las piezas se pueden testear por separado.** `paquete/tests/test_limpiar.py`
   verifica cada regla sin cargar el dataset completo.
3. **Las dependencias se declaran en `pyproject.toml`**, no en un `!pip
   install` a mitad del notebook.

---

## El resumen que hay que llevarse

| | Caotico | Script | Paquete |
|---|---|---|---|
| El resultado depende del orden de Shift+Enter | **si** | no | no |
| Las constantes estan en un solo sitio | no | si | si, y compartido entre modulos |
| Se puede importar una parte para probarla | no | no | **si** |
| Se puede testear | no | apenas | **si** (`pytest paquete/tests/`) |
| Las dependencias estan declaradas | no | implicito | **si** (`pyproject.toml`) |
| Otra persona lo reproduce en 1 comando | no | si (`python ...`) | **si** (`uv pip install -e paquete/`) |

El paso del script al paquete **no es cosmetico**: habilita cosas que no se
podian hacer antes. Y el paso del notebook caotico al script es la primera
pieza de disciplina que te saca de la zona peligrosa.

Siguiente paso: aplicalo a tu propio notebook. Ver [`taller.md`](taller.md).